# FLUX v6.0 Integration — Full Capability Test & Migration

**Date:** April 1, 2026  
**Goal:** Test all model capabilities, fix tool calling, embed codebase, cleanup

**Supported Environments:** Kaggle (T4/P100), Colab (T4), Local

## What This Notebook Does

1. **Test All Components** — CSE, Field, Memory, Decoder, Adapters, VLM, Causal
2. **Fix Tool Calling** — Convert to Qwen2.5 native JSON function format (no fine-tuning)
3. **Embed Codebase** — Make .flx self-contained with runtime code
4. **Cleanup** — Remove deprecated phases (1.5, 2.5, 3.5, 9) and old tool tags
5. **Full Integration Test** — End-to-end VLM + FLUX tools
6. **Save v6.0** — Autonomous, self-contained model

## Setup Instructions

### Kaggle
1. Add `HF_TOKEN` to Secrets (Add-ons → Secrets)
2. Enable GPU accelerator (T4 recommended)
3. Run all cells

### Colab
1. Run Cell 1 — it will clone the repository
2. Enter HF token when prompted (or set `HF_TOKEN` env var)
3. Enable GPU runtime (Runtime → Change runtime type → T4)

### Local
1. Ensure you're in the FLUX repository root
2. Set `HF_TOKEN` env var or create `.env` file with `HF_TOKEN=your_token`
3. Run all cells

## Repository & Model Sources

- **GitHub:** `https://github.com/Unseengap/FLUX.git`
- **HuggingFace:** `UnseenGAP/FLUX` → `checkpoints/Flux-Apex-V1.flx`

In [ ]:
"""Cell 1: Environment Detection & Repository Setup"""

import os
import sys
from pathlib import Path

# ════════════════════════════════════════════════════════════════════
# Environment Detection
# ════════════════════════════════════════════════════════════════════

GITHUB_REPO_URL = "https://github.com/Unseengap/FLUX.git"
HF_REPO_ID = "UnseenGAP/FLUX"

if Path('/kaggle').exists():
    ROOT = Path('/kaggle/working/FLUX')
    ENVIRONMENT = 'kaggle'
    if not ROOT.exists():
        print(f"Cloning repository to {ROOT}...")
        os.system(f'git clone {GITHUB_REPO_URL} {ROOT}')
elif Path('/content').exists():
    ROOT = Path('/content/FLUX')
    ENVIRONMENT = 'colab'
    if not ROOT.exists():
        print(f"Cloning repository to {ROOT}...")
        os.system(f'git clone {GITHUB_REPO_URL} {ROOT}')
else:
    ROOT = Path('/Users/admin/Desktop/flux')
    ENVIRONMENT = 'local'

print(f"Environment: {ENVIRONMENT.upper()}")
print(f"Root: {ROOT}")

# ════════════════════════════════════════════════════════════════════
# Python Path Setup
# ════════════════════════════════════════════════════════════════════

paths_to_add = [
    ROOT,
    ROOT / 'phases' / 'phase1',
    ROOT / 'phases' / 'phase2',
    ROOT / 'phases' / 'phase_orchestrator',
]

for p in paths_to_add:
    if str(p) not in sys.path and p.exists():
        sys.path.insert(0, str(p))

os.chdir(ROOT)
print(f"✓ Working directory: {Path.cwd()}")

# ════════════════════════════════════════════════════════════════════
# Install Dependencies (Kaggle/Colab only)
# ════════════════════════════════════════════════════════════════════

if ENVIRONMENT in ['kaggle', 'colab']:
    print("\nInstalling dependencies...")
    os.system(f'pip install -q -r {ROOT}/requirements.txt')
    os.system(f'pip install -q -e {ROOT}')
    print("✓ Dependencies installed")

# ════════════════════════════════════════════════════════════════════
# Core Imports
# ════════════════════════════════════════════════════════════════════

import gc
import torch
import torch.nn as nn
import json
import re
from datetime import datetime
from typing import Dict, Any, List, Optional

print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("GPU: Apple Silicon (MPS)")

print("\n✓ Environment setup complete")

In [ ]:
"""Cell 2: Initialize Logger & Resolve HuggingFace Token"""

from flux_utils import PhaseLogger, get_device, _resolve_hf_token
from flux_model import FLUXModel

log = PhaseLogger(phase=60)  # Phase 6.0 = Autonomous Integration
log.separator("FLUX v6.0 Integration — Full Capability Test")

device = get_device()
log.info(f"Device: {device}")

# ════════════════════════════════════════════════════════════════════
# HuggingFace Token Resolution
# Priority: Kaggle secrets → HF_TOKEN env var → .env file → HF cache
# ════════════════════════════════════════════════════════════════════

try:
    hf_token = _resolve_hf_token()
    log.success("HuggingFace token resolved")
except Exception as e:
    hf_token = None
    log.warning(f"No HF token found (public models still work): {e}")

print("\n✓ Logger initialized")

In [ ]:
"""Cell 3: Load Model using FLUXModel.load() — per spec"""

log.cell_start("Cell 3 — Load Model")

from huggingface_hub import hf_hub_download

# ════════════════════════════════════════════════════════════════════
# Model Path Setup
# ════════════════════════════════════════════════════════════════════

CHECKPOINT_DIR = ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = CHECKPOINT_DIR / 'Flux-Apex-V1.flx'

# ════════════════════════════════════════════════════════════════════
# Download from HuggingFace if not present
# ════════════════════════════════════════════════════════════════════

if not MODEL_PATH.exists():
    log.info("Model not found locally, downloading from HuggingFace...")
    log.info(f"  Repo: {HF_REPO_ID}")
    log.info(f"  File: checkpoints/Flux-Apex-V1.flx")
    
    try:
        downloaded_path = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename='checkpoints/Flux-Apex-V1.flx',
            token=hf_token,
            local_dir=str(ROOT),
            local_dir_use_symlinks=False,
        )
        MODEL_PATH = Path(downloaded_path)
        log.success(f"Downloaded to: {MODEL_PATH}")
    except Exception as e:
        raise FileNotFoundError(
            f"Failed to download model from HuggingFace: {e}\n"
            f"Ensure {HF_REPO_ID} exists and contains checkpoints/Flux-Apex-V1.flx"
        )
else:
    log.success(f"Model found locally: {MODEL_PATH}")

# ════════════════════════════════════════════════════════════════════
# Load Model using FLUXModel.load() — per copilot-instructions.md
# ════════════════════════════════════════════════════════════════════

log.info("Loading model using FLUXModel.load()...")
model = FLUXModel.load(str(MODEL_PATH))

# ════════════════════════════════════════════════════════════════════
# Model Inspection
# ════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print(f"MODEL INSPECTION")
print(f"{'='*60}")
print(f"Format: {model.state.get('format', 'unknown')}")
print(f"Version: {model.version}")
print(f"Phase: {model.phase}")
print(f"File size: {MODEL_PATH.stat().st_size / 1e9:.2f} GB")

# Parameter count
total_params = 0
print(f"\nComponents ({len(model.state)} keys):")
for key in sorted(model.state.keys()):
    data = model.state[key]
    if isinstance(data, dict):
        if 'state_dict' in data:
            sd = data['state_dict']
            if isinstance(sd, dict):
                params = sum(v.numel() for v in sd.values() if isinstance(v, torch.Tensor))
                total_params += params
                print(f"  {key}: {params:,} params")
        elif 'weights' in data:
            w = data['weights']
            params = sum(v.numel() for v in w.values() if isinstance(v, torch.Tensor))
            total_params += params
            print(f"  {key}: {params:,} params (weights)")

print(f"\nTotal parameters: {total_params:,}")
print(f"Active components: {model.active_components}")

# Check for legacy components
legacy = model.get_legacy_components()
if legacy:
    print(f"\nLegacy components (marked for removal):")
    for comp in legacy:
        info = model.get_legacy_info(comp)
        print(f"  ⚠ {comp}: {info.get('reason', 'no reason')}")

log.cell_end("Cell 3 — Load Model", "PASS")

In [ ]:
"""Cell 4: Test CSE (Continuous Semantic Encoder)"""

log.cell_start("Cell 4 — Test CSE")

from cse import ContinuousSemanticEncoder

# Load CSE using FLUXModel.get_component()
cse_state = model.get_component('cse')
if cse_state is None:
    cse_state = model.state.get('cse')

CSE_TEST_PASSED = False

if cse_state is None:
    log.error("CSE component not found in model")
else:
    try:
        cse_config = cse_state.get('config', {})
        
        # Initialize CSE with wave_dims dict
        # If wave_dims not in config, CSE __init__ will use defaults
        wave_dims = cse_config.get('wave_dims', None)
        
        cse = ContinuousSemanticEncoder(wave_dims=wave_dims)
        
        # Load weights if available
        cse_sd = cse_state.get('state_dict')
        if cse_sd:
            cse.load_state_dict(cse_sd, strict=False)
            log.success("CSE state_dict loaded")
        else:
            log.warning("CSE has no state_dict, using fresh weights")
        
        cse.to(device)
        cse.eval()
        
        log.success("CSE initialized via model.get_component('cse')")
        
        # Test encoding
        test_texts = [
            "Hello, world!",
            "The quick brown fox jumps over the lazy dog.",
            "FLUX uses physics-inspired components for AI.",
            "ARC puzzles test abstract reasoning.",
        ]
        
        print("\nCSE Encoding Test:")
        print("-" * 50)
        
        cse_results = {}
        with torch.no_grad():
            for text in test_texts:
                wave = cse.encode(text)
                cse_results[text] = wave
                print(f"  '{text[:30]}...' → shape: {wave.shape}")
        
        # Test semantic similarity
        def cosine_sim(a, b):
            a_flat = a.flatten()
            b_flat = b.flatten()
            return torch.cosine_similarity(a_flat.unsqueeze(0), b_flat.unsqueeze(0)).item()
        
        w1 = cse_results[test_texts[2]]  # FLUX
        w2 = cse_results[test_texts[3]]  # ARC
        w3 = cse_results[test_texts[0]]  # Hello
        
        print(f"\nSemantic Similarity (cosine):")
        print(f"  FLUX vs ARC: {cosine_sim(w1, w2):.4f}")
        print(f"  FLUX vs Hello: {cosine_sim(w1, w3):.4f}")
        
        # PASS if wave dimension is 432
        CSE_TEST_PASSED = w1.shape[-1] == 432
        
    except Exception as e:
        log.error(f"CSE test failed: {e}")
        import traceback
        traceback.print_exc()

log.success(f"CSE test: {'PASS' if CSE_TEST_PASSED else 'FAIL'}")
log.cell_end("Cell 4 — Test CSE", "PASS" if CSE_TEST_PASSED else "FAIL")

In [ ]:
"""Cell 5: Test Resonance Field"""

log.cell_start("Cell 5 — Test Field")

from resonance_field import ResonanceField

# Load field using FLUXModel.get_component()
field_state = model.get_component('field')
if field_state is None:
    field_state = model.state.get('field')

FIELD_TEST_PASSED = False

if field_state is None:
    log.error("Field component not found in model")
else:
    try:
        field_config = field_state.get('config', {})
        
        # Get dimensions (handle various naming conventions)
        H = field_config.get('h') or field_config.get('H') or field_config.get('height') or 96
        W = field_config.get('w') or field_config.get('W') or field_config.get('width') or 96
        D = field_config.get('d') or field_config.get('D') or field_config.get('depth') or 96
        F = field_config.get('features') or field_config.get('F') or field_config.get('feature_dim') or 512
        
        log.success(f"Field dimensions: {H}×{W}×{D}×{F}")
        
        # Initialize field
        field = ResonanceField(H=H, W=W, D=D, features=F)
        
        # Load state - handle various structures
        field_sd = field_state.get('state_dict', {})
        
        if isinstance(field_sd, dict):
            if 'state' in field_sd:
                # Direct field tensor stored as 'state'
                state_tensor = field_sd['state']
                if isinstance(state_tensor, torch.Tensor):
                    field.field.data = state_tensor
                    print(f"✓ Field state loaded: {state_tensor.shape}")
            elif 'field' in field_sd:
                # Stored under 'field' key
                state_tensor = field_sd['field']
                if isinstance(state_tensor, torch.Tensor):
                    field.field.data = state_tensor
                    print(f"✓ Field data loaded: {state_tensor.shape}")
            elif len(field_sd) > 0:
                # Try to load the whole state_dict
                try:
                    field.load_state_dict(field_sd, strict=False)
                    print(f"✓ Field state_dict loaded ({len(field_sd)} keys)")
                except Exception as e:
                    print(f"⚠ Could not load field state_dict: {e}")
        
        field.to(device)
        
        # Field statistics
        print(f"\nField statistics:")
        print(f"  Shape: {field.field.shape}")
        print(f"  Mean: {field.field.data.mean().item():.6f}")
        print(f"  Std: {field.field.data.std().item():.6f}")
        print(f"  Non-zero: {(field.field.data.abs() > 1e-6).float().mean().item():.2%}")
        
        FIELD_TEST_PASSED = field.field.shape == torch.Size([H, W, D, F])
        
    except Exception as e:
        log.error(f"Field test failed: {e}")
        import traceback
        traceback.print_exc()

log.success(f"Field test: {'PASS' if FIELD_TEST_PASSED else 'FAIL'}")
log.cell_end("Cell 5 — Test Field", "PASS" if FIELD_TEST_PASSED else "FAIL")

In [ ]:
"""Cell 6: Test Memory System"""

log.cell_start("Cell 6 — Test Memory")

# Memory components are stored at TOP-LEVEL in v7.x, not in nested 'memory' dict
# Check: working_memory, episodic_memory, semantic_memory, spatial_memory

print("Memory System Components:")
print("-" * 50)

# Check for top-level memory components (v7.x structure)
working_memory = model.state.get('working_memory') or model.get_component('working_memory')
episodic_memory = model.state.get('episodic_memory') or model.get_component('episodic_memory')
semantic_memory = model.state.get('semantic_memory') or model.get_component('semantic_memory')
spatial_memory = model.state.get('spatial_memory')

# Also check nested 'memory' dict (legacy structure)
memory_state = model.get_component('memory') or model.state.get('memory', {})

found_memory = False

# Working memory
if working_memory:
    print(f"  ✓ Working memory: present")
    if isinstance(working_memory, dict):
        print(f"    Keys: {list(working_memory.keys())[:5]}")
    found_memory = True
elif 'working' in memory_state:
    working = memory_state['working']
    print(f"  ✓ Working memory (nested):")
    print(f"    Window size: {working.get('window_size', 'N/A')}")
    found_memory = True

# Episodic memory
if episodic_memory:
    print(f"  ✓ Episodic memory: present")
    if isinstance(episodic_memory, dict) and 'vectors' in episodic_memory:
        vecs = episodic_memory['vectors']
        if isinstance(vecs, torch.Tensor):
            print(f"    Vector shape: {vecs.shape}")
    found_memory = True
elif 'episodic' in memory_state:
    episodic = memory_state['episodic']
    print(f"  ✓ Episodic memory (nested):")
    print(f"    Entries: {len(episodic.get('metadata', []))}")
    found_memory = True

# Semantic memory
if semantic_memory:
    print(f"  ✓ Semantic memory: present")
    found_memory = True
elif 'semantic' in memory_state:
    print(f"  ✓ Semantic memory (nested): present")
    found_memory = True

# Spatial memory (Ice/Water curiosity)
if spatial_memory:
    print(f"  ✓ Spatial memory: present")
    if isinstance(spatial_memory, dict):
        state_dict = spatial_memory.get('state_dict', spatial_memory)
        if isinstance(state_dict, dict):
            if 'exploration_mass' in state_dict:
                em = state_dict['exploration_mass']
                if isinstance(em, torch.Tensor):
                    explored = (em > 0).sum().item()
                    print(f"    Explored cells: {explored}")
            if 'curiosity_field' in state_dict:
                cf = state_dict['curiosity_field']
                if isinstance(cf, torch.Tensor):
                    curious = (cf > 0).sum().item()
                    print(f"    Curiosity cells: {curious}")
    found_memory = True

if not found_memory:
    print("  ⚠ No memory components found")

MEMORY_TEST_PASSED = found_memory
log.success(f"Memory test: {'PASS' if MEMORY_TEST_PASSED else 'FAIL'}")
log.cell_end("Cell 6 — Test Memory", "PASS" if MEMORY_TEST_PASSED else "FAIL")

In [ ]:
"""Cell 7: Test Adapters (Grid↔Wave)"""

log.cell_start("Cell 7 — Test Adapters")

# Check multiple locations for adapters:
# 1. model.state['adapters'] - nested dict (v4.x+)
# 2. model.state['grid_adapters'] - may have been injected
# 3. model.state['grid_to_wave'] - direct component

adapters_state = model.state.get('adapters', {})
grid_adapters = model.state.get('grid_adapters', {})

print("Available Adapters:")
print("-" * 50)

# Check nested adapters
if adapters_state:
    for name, data in adapters_state.items():
        if isinstance(data, dict):
            if 'state_dict' in data:
                params = sum(v.numel() for v in data['state_dict'].values() if isinstance(v, torch.Tensor))
                print(f"  adapters.{name}: {params:,} params")
            else:
                print(f"  adapters.{name}: config only")

# Check grid_adapters (may be injected from gridtowave_contrastive.pt)
if grid_adapters:
    print(f"\nGrid Adapters:")
    if 'encoder' in grid_adapters:
        enc = grid_adapters['encoder']
        if isinstance(enc, dict):
            params = sum(v.numel() for v in enc.values() if isinstance(v, torch.Tensor))
            print(f"  ✓ grid_adapters.encoder (GridToWave): {params:,} params")
    if 'decoder' in grid_adapters:
        dec = grid_adapters['decoder']
        if isinstance(dec, dict):
            params = sum(v.numel() for v in dec.values() if isinstance(v, torch.Tensor))
            print(f"  ✓ grid_adapters.decoder (WaveToGrid): {params:,} params")

# Determine which GridToWave to use
g2w_state = None
g2w_source = None

if grid_adapters and 'encoder' in grid_adapters:
    g2w_state = {'state_dict': grid_adapters['encoder'], 'config': grid_adapters.get('config', {})}
    g2w_source = 'grid_adapters.encoder'
elif adapters_state.get('grid_to_wave'):
    g2w_state = adapters_state.get('grid_to_wave')
    g2w_source = 'adapters.grid_to_wave'
elif model.state.get('grid_to_wave'):
    g2w_state = model.state.get('grid_to_wave')
    g2w_source = 'grid_to_wave'

ADAPTER_TEST_PASSED = False

if g2w_state:
    print(f"\nUsing GridToWave from: {g2w_source}")
    try:
        # Try importing from phase8_8 (GridToWaveEncoder) or phase8 (GridToWave)
        try:
            sys.path.insert(0, str(ROOT / 'phases' / 'phase8_8'))
            from grid_to_wave import GridToWaveEncoder
            encoder_class = GridToWaveEncoder
        except ImportError:
            sys.path.insert(0, str(ROOT / 'phases' / 'phase8'))
            from grid_adapters import GridToWave
            encoder_class = GridToWave
        
        g2w_config = g2w_state.get('config', {})
        g2w = encoder_class(
            wave_dim=g2w_config.get('wave_dim', 432),
            max_grid_size=g2w_config.get('max_grid_size', g2w_config.get('max_size', 30)),
        )
        
        if 'state_dict' in g2w_state:
            try:
                g2w.load_state_dict(g2w_state['state_dict'], strict=False)
                print(f"  ✓ Loaded state_dict")
            except Exception as e:
                print(f"  ⚠ Could not load state_dict: {e}")
        
        g2w.to(device)
        g2w.eval()
        
        # Test encoding
        test_grid = [[0, 1, 0], [1, 2, 1], [0, 1, 0]]
        with torch.no_grad():
            # GridToWave uses .encode() method, not __call__
            if hasattr(g2w, 'encode'):
                grid_wave = g2w.encode(test_grid, mode='holistic')
            else:
                grid_tensor = torch.tensor(test_grid, dtype=torch.long, device=device)
                grid_wave = g2w(grid_tensor.unsqueeze(0))
        
        print(f"\nGrid→Wave test:")
        print(f"  Input: 3×3 grid → Output: {grid_wave.shape}")
        ADAPTER_TEST_PASSED = grid_wave.shape[-1] == 432
        
    except Exception as e:
        print(f"  Grid→Wave test failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print("\n⚠ No grid_to_wave adapter found")
    print("  To inject: load gridtowave_contrastive.pt into grid_adapters.encoder")

log.success(f"Adapter test: {'PASS' if ADAPTER_TEST_PASSED else 'PARTIAL'}")
log.cell_end("Cell 7 — Test Adapters", "PASS" if ADAPTER_TEST_PASSED else "PARTIAL")

In [ ]:
"""Cell 8: Test Causal System (CGN + Graph + Tracker + Rules)"""

log.cell_start("Cell 8 — Test Causal")

print("Causal System:")
print("-" * 50)

# Check multiple locations:
# 1. model.state['causal'] - nested structure (v4.x)
# 2. model.state['causal_tracker'] - top-level (v7.x)
# 3. model.state['learned_rules'] - top-level (v7.x)
# 4. model.state['state']['causal_tracker'] - nested state dict

causal_state = model.state.get('causal', {})
causal_tracker = model.state.get('causal_tracker')
learned_rules = model.state.get('learned_rules')

# Also check nested 'state' dict
nested_state = model.state.get('state', {})
if not causal_tracker:
    causal_tracker = nested_state.get('causal_tracker')
if not learned_rules:
    learned_rules = nested_state.get('learned_rules')

found_causal = False

# CGN nodes (from causal.cgn_state)
if 'cgn_state' in causal_state:
    cgn = causal_state['cgn_state']
    node_count = cgn.get('node_count', len(cgn.get('nodes', [])))
    print(f"  ✓ CGN nodes: {node_count}")
    if 'manifolds' in cgn:
        print(f"    Manifolds: {len(cgn['manifolds'])}")
    found_causal = True

# Causal graph (from causal.graph_state)
if 'graph_state' in causal_state:
    graph = causal_state['graph_state']
    print(f"  ✓ Causal graph:")
    print(f"    Links: {len(graph.get('links', []))}")
    print(f"    Nodes: {len(graph.get('nodes', []))}")
    found_causal = True

# Causal tracker (v7.x top-level structure)
if causal_tracker:
    print(f"\n  ✓ Causal Tracker:")
    if isinstance(causal_tracker, dict):
        links = causal_tracker.get('links', [])
        total = causal_tracker.get('total', len(links))
        print(f"    Links stored: {len(links)}")
        print(f"    Total recorded: {total}")
        
        if links and len(links) > 0:
            # Show sample link
            sample = links[0]
            print(f"    Sample: {str(sample)[:60]}...")
    found_causal = True

# Learned rules (v7.x top-level structure)
if learned_rules:
    print(f"\n  ✓ Learned Rules:")
    if isinstance(learned_rules, dict):
        rules = learned_rules.get('rules', [])
        total = learned_rules.get('total', len(rules))
        print(f"    Rules: {len(rules)}")
        print(f"    Total: {total}")
        
        if rules and len(rules) > 0:
            print(f"    Sample rules:")
            for i, rule in enumerate(rules[:3]):
                print(f"      {i+1}. {str(rule)[:70]}...")
    found_causal = True

if not found_causal:
    print("  ⚠ No causal components found")

CAUSAL_TEST_PASSED = found_causal
log.success(f"Causal test: {'PASS' if CAUSAL_TEST_PASSED else 'FAIL'}")
log.cell_end("Cell 8 — Test Causal", "PASS" if CAUSAL_TEST_PASSED else "FAIL")

In [ ]:
"""Cell 9: Test Embedded Models (VLM & Computer Vision)"""

log.cell_start("Cell 9 — Test Embedded Models")

# Check multiple locations for embedded models:
# 1. model.state['models'] - contains face, object_detect, depth, pose, etc.
# 2. model.state['vlm'] - embedded VLM (Qwen2.5-VL)

models_state = model.state.get('models', {})
vlm_state = model.state.get('vlm')

print("Embedded Models:")
print("-" * 50)

total_model_params = 0
embedded_models = []

# Check models dict (computer vision models)
if models_state:
    print("\n[Computer Vision Models]")
    for name, state in models_state.items():
        if not isinstance(state, dict):
            continue
        params = state.get('total_params', 0)
        total_model_params += params
        has_weights = bool(state.get('weights') or state.get('state') or state.get('onnx_models'))
        lazy = 'lazy' if state.get('lazy_load', True) else 'always'
        icon = '✓' if has_weights else '○'
        
        if has_weights:
            embedded_models.append(f"models.{name}")
        
        size_gb = params * 2 / 1e9 if params > 0 else 0
        print(f"  {icon} {name:15}: {params:>12,} params ({size_gb:>5.2f} GB) [{lazy}]")

# Check VLM (embedded Qwen2.5-VL)
if vlm_state:
    print("\n[Vision-Language Model (VLM)]")
    if isinstance(vlm_state, dict):
        base_model = vlm_state.get('base_model', 'unknown')
        vlm_params = vlm_state.get('total_params', 0)
        has_weights = bool(vlm_state.get('weights'))
        
        # Count weights if total_params not stored
        if vlm_params == 0 and vlm_state.get('weights'):
            weights = vlm_state['weights']
            vlm_params = sum(v.numel() for v in weights.values() if isinstance(v, torch.Tensor))
        
        total_model_params += vlm_params
        
        icon = '✓' if has_weights else '○'
        size_gb = vlm_params * 2 / 1e9
        print(f"  {icon} VLM: {base_model}")
        print(f"    Parameters: {vlm_params:,} ({size_gb:.2f} GB)")
        print(f"    Weights embedded: {has_weights}")
        
        if has_weights:
            embedded_models.append("vlm")
        
        # Check for bridges
        if 'bridges' in vlm_state:
            bridges = vlm_state['bridges']
            print(f"    Wave↔VLM bridges:")
            for bname, bdata in bridges.items():
                print(f"      {bname}: {bdata}")

# Check metadata for vlm_embedded flag
metadata = model.state.get('metadata', {})
if metadata.get('vlm_embedded'):
    print(f"\n  ✓ VLM marked as embedded in metadata")
    print(f"    Base model: {metadata.get('vlm_base_model', 'unknown')}")

print(f"\n" + "-" * 50)
print(f"Total model params: {total_model_params:,}")
print(f"Embedded models: {len(embedded_models)}")

MODELS_TEST_PASSED = len(embedded_models) > 0 or (vlm_state and vlm_state.get('weights'))
log.success(f"Embedded models test: {'PASS' if MODELS_TEST_PASSED else 'FAIL'}")
log.cell_end("Cell 9 — Test Embedded Models", "PASS" if MODELS_TEST_PASSED else "FAIL")

In [ ]:
"""Cell 10: Define Native JSON Tools (Qwen2.5 Function Calling Format)"""

log.cell_start("Cell 10 — Define Tools")

# Qwen2.5 native function calling format — no fine-tuning needed
FLUX_TOOLS_JSON = [
    # ═══ PERCEPTION ═══
    {
        "type": "function",
        "function": {
            "name": "encode_text",
            "description": "Encode text into 432-dimensional semantic waves using CSE.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "Text to encode (UTF-8)"}
                },
                "required": ["text"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "encode_grid",
            "description": "Encode ARC grid (colors 0-9, up to 30x30) into wave representation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "grid": {"type": "array", "items": {"type": "array", "items": {"type": "integer"}}},
                    "mode": {"type": "string", "enum": ["holistic", "spatial"], "default": "holistic"}
                },
                "required": ["grid"]
            }
        }
    },
    # ═══ KNOWLEDGE ═══
    {
        "type": "function",
        "function": {
            "name": "query_field",
            "description": "Search the 96³ resonance field for relevant knowledge.",
            "parameters": {
                "type": "object",
                "properties": {
                    "wave_id": {"type": "string", "description": "Wave ID or '$LAST'"},
                    "top_k": {"type": "integer", "default": 5}
                },
                "required": ["wave_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "recall_memory",
            "description": "Search episodic memory for facts.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "limit": {"type": "integer", "default": 5}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "store_memory",
            "description": "Store a fact in episodic memory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "content": {"type": "string"},
                    "importance": {"type": "number", "minimum": 0, "maximum": 1, "default": 0.5}
                },
                "required": ["content"]
            }
        }
    },
    # ═══ REASONING ═══
    {
        "type": "function",
        "function": {
            "name": "predict_effect",
            "description": "Predict causal effect of an action.",
            "parameters": {
                "type": "object",
                "properties": {
                    "action": {"type": "string"},
                    "target": {"type": "string"}
                },
                "required": ["action", "target"]
            }
        }
    },
    # ═══ GENERATION ═══
    {
        "type": "function",
        "function": {
            "name": "decode_grid",
            "description": "Convert wave back to ARC grid.",
            "parameters": {
                "type": "object",
                "properties": {
                    "wave_id": {"type": "string"},
                    "height": {"type": "integer"},
                    "width": {"type": "integer"}
                },
                "required": ["wave_id", "height", "width"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_code",
            "description": "Execute Python code for precise calculations.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {"type": "string"},
                    "timeout": {"type": "integer", "default": 30}
                },
                "required": ["code"]
            }
        }
    },
]

print(f"Defined {len(FLUX_TOOLS_JSON)} tools in native JSON format")
for tool in FLUX_TOOLS_JSON:
    print(f"  • {tool['function']['name']}")

log.cell_end("Cell 10 — Define Tools", "PASS")

In [ ]:
"""Cell 11: Create System Prompt (v6.0)"""

log.cell_start("Cell 11 — System Prompt")

FLUX_SYSTEM_PROMPT_V6 = """\
You are FLUX, a physics-inspired cognitive AI using resonance fields, semantic waves, and gravitational relevance.

## Capabilities

**Perception**
- `encode_text`: Text → 432D semantic wave
- `encode_grid`: ARC grid → wave representation

**Knowledge**
- `query_field`: Search the 96³ resonance field
- `recall_memory`: Search episodic memory
- `store_memory`: Remember important facts

**Reasoning**
- `predict_effect`: Causal prediction

**Generation**
- `decode_grid`: Wave → ARC grid
- `run_code`: Execute Python

## ARC Strategy

1. `encode_grid` input → wave
2. `query_field` for patterns
3. `predict_effect` to test hypothesis
4. `decode_grid` to produce output

Always explain your reasoning.
"""

print(f"System prompt created: {len(FLUX_SYSTEM_PROMPT_V6)} chars")
log.cell_end("Cell 11 — System Prompt", "PASS")

In [ ]:
"""Cell 12: Bundle Codebase for Embedding"""

log.cell_start("Cell 12 — Bundle Codebase")

import base64
import gzip

ESSENTIAL_FILES = [
    'flux_model.py',
    'flux_utils.py',
    'phases/phase1/cse.py',
    'phases/phase1/wave_types.py',
    'phases/phase2/resonance_field.py',
    'phases/phase2/flux_format.py',
    'phases/phase3/gravitational_relevance.py',
    'phases/phase5/causal_geometry_node.py',
    'phases/phase6/episodic_memory.py',
    'phases/phase8_8/grid_to_wave.py',
]

codebase_bundle = {
    'version': '6.0',
    'timestamp': datetime.now().isoformat(),
    'files': {},
    'total_uncompressed': 0,
    'total_compressed': 0,
}

print("Bundling codebase:")
for rel_path in ESSENTIAL_FILES:
    full_path = ROOT / rel_path
    if full_path.exists():
        try:
            code = full_path.read_text(encoding='utf-8')
            compressed = gzip.compress(code.encode('utf-8'))
            encoded = base64.b64encode(compressed).decode('ascii')
            codebase_bundle['files'][rel_path] = encoded
            codebase_bundle['total_uncompressed'] += len(code)
            codebase_bundle['total_compressed'] += len(compressed)
            print(f"  ✓ {rel_path}: {len(code):,} → {len(compressed):,} bytes")
        except Exception as e:
            print(f"  ✗ {rel_path}: {e}")
    else:
        print(f"  - {rel_path}: not found")

ratio = codebase_bundle['total_compressed'] / max(codebase_bundle['total_uncompressed'], 1)
print(f"\nBundle: {len(codebase_bundle['files'])} files, {ratio:.1%} compression")

log.cell_end("Cell 12 — Bundle Codebase", "PASS")

In [ ]:
"""Cell 13: Update Model with v6.0 Changes"""

log.cell_start("Cell 13 — Update Model")

# ════════════════════════════════════════════════════════════════════
# Update model using FLUXModel interface per copilot-instructions.md
# ════════════════════════════════════════════════════════════════════

# 1. Update version
model.version = '6.0-autonomous'
model.phase = 'autonomous'

# 2. Add orchestration with native JSON tools
model.state['orchestration'] = {
    'tools': FLUX_TOOLS_JSON,
    'tool_format': 'native_json',
    'system_prompt': FLUX_SYSTEM_PROMPT_V6,
    'version': '2.0',
    'capabilities': ['text_encoding', 'grid_encoding', 'field_query', 'memory_ops', 'causal_reasoning', 'code_execution'],
}
model.components['orchestration'] = True

# 3. Embed runtime codebase
model.state['runtime'] = {
    'files': codebase_bundle['files'],
    'version': codebase_bundle['version'],
    'timestamp': codebase_bundle['timestamp'],
    'total_bytes': codebase_bundle['total_compressed'],
}
model.components['runtime_embedded'] = True

# 4. Update metadata
model.metadata['last_modified'] = datetime.now().isoformat()
model.metadata['modified_components'] = ['orchestration', 'runtime']
model.metadata['autonomous'] = True
model.metadata['tool_format'] = 'native_json'

# 5. Update component flags
model.components['tool_use'] = True
model._modified = True

print(f"Model updated:")
print(f"  Version: {model.version}")
print(f"  Phase: {model.phase}")
print(f"  Tools: {len(model.state['orchestration']['tools'])}")
print(f"  Runtime files: {len(model.state['runtime']['files'])}")
print(f"  Active components: {model.active_component_count}")

log.cell_end("Cell 13 — Update Model", "PASS")

In [ ]:
"""Cell 14: Save Model using model.save() — per spec"""

log.cell_start("Cell 14 — Save Model")

import shutil

OUTPUT_PATH = ROOT / 'checkpoints' / 'Flux-Apex-V1.flx'
BACKUP_PATH = ROOT / 'checkpoints' / 'Flux-Apex-V1-backup.flx'

# Create backup
if OUTPUT_PATH.exists():
    shutil.copy(OUTPUT_PATH, BACKUP_PATH)
    log.info(f"Backup created: {BACKUP_PATH}")

# ════════════════════════════════════════════════════════════════════
# Save using model.save() — "Save as LAST STEP (ALWAYS)"
# ════════════════════════════════════════════════════════════════════

log.info("Saving model using model.save()...")
model.save(str(OUTPUT_PATH), overwrite=True)

# Verify
final_size = OUTPUT_PATH.stat().st_size / 1e9
log.success(f"Saved: {final_size:.2f} GB")

# Verify by re-loading
log.info("Verifying saved model...")
verify_model = FLUXModel.load(str(OUTPUT_PATH))
print(f"  Version: {verify_model.version}")
print(f"  Tools: {len(verify_model.state.get('orchestration', {}).get('tools', []))}")
print(f"  Runtime: {'YES' if 'runtime' in verify_model.state else 'NO'}")

SAVE_VERIFIED = (
    verify_model.version == '6.0-autonomous' and
    len(verify_model.state.get('orchestration', {}).get('tools', [])) > 0
)
del verify_model

log.success(f"Save verification: {'PASS' if SAVE_VERIFIED else 'FAIL'}")
log.cell_end("Cell 14 — Save Model", "PASS" if SAVE_VERIFIED else "FAIL")

In [ ]:
"""Cell 15: Upload to HuggingFace (Optional)"""

log.cell_start("Cell 15 — Upload")

if hf_token:
    from huggingface_hub import HfApi
    
    log.info("Uploading to HuggingFace Hub...")
    
    try:
        api = HfApi(token=hf_token)
        api.upload_file(
            path_or_fileobj=str(OUTPUT_PATH),
            path_in_repo='checkpoints/Flux-Apex-V1.flx',
            repo_id=HF_REPO_ID,
            commit_message=f'v{model.version} - autonomous integration',
        )
        log.success("Uploaded to HuggingFace Hub")
    except Exception as e:
        log.error(f"Upload failed: {e}")
else:
    log.warning("No HF token - skipping upload")
    print("  Upload manually with:")
    print("    from flux_utils import upload_flx_to_hf")
    print("    upload_flx_to_hf('checkpoints/Flux-Apex-V1.flx')")

log.cell_end("Cell 15 — Upload", "PASS")

In [ ]:
"""Cell 16: Final Summary"""

log.separator("FLUX v6.0 INTEGRATION COMPLETE")

tests = {
    'CSE': CSE_TEST_PASSED,
    'Field': FIELD_TEST_PASSED,
    'Memory': MEMORY_TEST_PASSED,
    'Adapters': ADAPTER_TEST_PASSED,
    'Causal': CAUSAL_TEST_PASSED,
    'Models': MODELS_TEST_PASSED,
    'Save': SAVE_VERIFIED,
}

print(f"""
╔══════════════════════════════════════════════════════════════╗
║             FLUX v6.0 INTEGRATION COMPLETE                   ║
╠══════════════════════════════════════════════════════════════╣
║  Output:   {str(OUTPUT_PATH):<47} ║
║  Size:     {final_size:.2f} GB{' ':<45}║
║  Version:  {model.version:<47} ║
╠══════════════════════════════════════════════════════════════╣""")

print("║  Test Results:" + " "*47 + "║")
for name, passed in tests.items():
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"║    {name:12}: {status:<45} ║")

total_passed = sum(tests.values())
total_tests = len(tests)

print(f"╠══════════════════════════════════════════════════════════════╣")
print(f"║  Total: {total_passed}/{total_tests} tests passed{' ':<42}║")
print(f"╠══════════════════════════════════════════════════════════════╣")
print(f"║  Usage:" + " "*53 + "║")
print(f"║    from flux_model import FLUXModel" + " "*25 + "║")
print(f"║    model = FLUXModel.load('checkpoints/Flux-Apex-V1.flx')" + " "*3 + "║")
print(f"╚══════════════════════════════════════════════════════════════╝")